In [1]:
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Activation, BatchNormalization
from tensorflow.keras.optimizers import RMSprop

In [2]:
filepath = tf.keras.utils.get_file('shakespeare.txt',origin='https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt')
text = open(filepath,'rb').read().decode(encoding='utf-8').lower()

In [3]:
text = text[:800000]

In [4]:
characters = sorted(set(text))
char_to_index = dict((c,i) for i,c in enumerate(characters))
index_to_char = dict((i,c) for i,c in enumerate(characters))

In [5]:
SEQ_LENGTH = 50
STEP_SIZE = 3
sentences = []
next_characters = []

In [6]:
for i in range(0,len(text)-SEQ_LENGTH,STEP_SIZE):
  sentences.append(text[i:i+SEQ_LENGTH])
  next_characters.append(text[i+SEQ_LENGTH])

In [7]:
x = np.zeros((len(sentences), SEQ_LENGTH, len(characters)), dtype=bool)
y = np.zeros((len(sentences), len(characters)), dtype=bool)

In [8]:
for i , sentence in enumerate(sentences):
  for t,character in enumerate(sentence):
    x[i,t,char_to_index[character]] = 1
  y[i,char_to_index[next_characters[i]]] = 1

In [10]:
# Ensure TensorFlow uses GPU if available
from tensorflow.keras.layers import GRU, Bidirectional
print("GPUs available:", tf.config.list_physical_devices('GPU'))

model = Sequential([
    #LSTM(512, input_shape=(SEQ_LENGTH, len(characters)), return_sequences=True),
    Bidirectional(LSTM(512, return_sequences=True), input_shape=(SEQ_LENGTH, len(characters))),
    Dropout(0.3),
    BatchNormalization(),

    LSTM(512),
    Dropout(0.3),
    BatchNormalization(),

    Dense(256, activation='relu'),
    Dropout(0.3),

    Dense(len(characters), activation='softmax')
])

optimizer = RMSprop(learning_rate=0.0005)
model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

model.summary()

# Train the model (automatically uses GPU if available)
history = model.fit(x, y, batch_size=256, epochs=20, validation_split=0.2)


GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 50, 1024)       │     2,260,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 50, 1024)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 50, 1024)       │         4,096 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 512)            │     3,147,776 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 39)             │        10,023 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,556,263 (21.20 MB)

 Trainable params: 5,553,191 (21.18 MB)

 Non-trainable params: 3,072 (12.00 KB)

Epoch 1/20
834/834 ━━━━━━━━━━━━━━━━━━━━ 148s 172ms/step - accuracy: 0.2812 - loss: 2.6115 - val_accuracy: 0.4192 - val_loss: 1.9717
Epoch 2/20
834/834 ━━━━━━━━━━━━━━━━━━━━ 144s 173ms/step - accuracy: 0.4282 - loss: 1.9230 - val_accuracy: 0.4621 - val_loss: 1.8096
Epoch 3/20
834/834 ━━━━━━━━━━━━━━━━━━━━ 144s 173ms/step - accuracy: 0.4702 - loss: 1.7577 - val_accuracy: 0.4754 - val_loss: 1.7629
Epoch 4/20
834/834 ━━━━━━━━━━━━━━━━━━━━ 144s 173ms/step - accuracy: 0.4945 - loss: 1.6679 - val_accuracy: 0.4928 - val_loss: 1.7164
Epoch 5/20
834/834 ━━━━━━━━━━━━━━━━━━━━ 144s 173ms/step - accuracy: 0.5065 - loss: 1.6110 - val_accuracy: 0.4976 - val_loss: 1.6906
Epoch 6/20
834/834 ━━━━━━━━━━━━━━━━━━━━ 144s 173ms/step - accuracy: 0.5220 - loss: 1.5612 - val_accuracy: 0.5028 - val_loss: 1.6656
Epoch 7/20
834/834 ━━━━━━━━━━━━━━━━━━━━ 144s 173ms/step - accuracy: 0.5304 - loss: 1.5258 - val_accuracy: 0.5105 - val_loss: 1.6355
Epoch 8/20
834/834 ━━━━━━━━━━━━━━━━━━━━ 144s 173ms/step - accuracy: 0.5360 -

In [11]:
model.save('textgenerator.keras')

In [12]:
def sample(preds,temprature=1.0):
  preds = np.asarray(preds).astype('float64')
  preds = np.log(preds) / temprature
  exp_preds = np.exp(preds)
  preds = exp_preds / np.sum(exp_preds)
  probas = np.random.multinomial(1,preds,1)
  return np.argmax(probas)

In [15]:
def generate_text(length, temperature=1.0):
    start_index = random.randint(0, len(text) - SEQ_LENGTH - 1)
    generated = ''
    sentence = text[start_index: start_index + SEQ_LENGTH]
    generated += sentence

    for i in range(length):
        x = np.zeros((1, SEQ_LENGTH, len(characters)))
        for t, character in enumerate(sentence):
            x[0, t, char_to_index[character]] = 1

        prediction = model.predict(x, verbose=0)[0]
        prediction = np.log(np.clip(prediction, 1e-8, 1.0)) / temperature
        prediction = np.exp(prediction) / np.sum(np.exp(prediction))

        # sample next character
        next_index = np.random.choice(len(prediction), p=prediction)
        next_character = index_to_char[next_index]

        generated += next_character
        sentence = sentence[1:] + next_character

    return generated


In [16]:
print('_______________0.2_______________')
print(generate_text(300,0.2))
print('_______________0.4_______________')
print(generate_text(300,0.4))
print('_______________0.6_______________')
print(generate_text(300,0.6))
print('_______________0.8_______________')
print(generate_text(300,0.8))

_______________0.2_______________
d at your highness' hands.
the language i have leave the sent.

sicinius:
i would the country's laster.

king richard ii:
i would the state, that i shall be the wife.

coriolanus:
i would the boots to should be the state,
the consul, and the stranger of the princes.

coriolanus:
i will be say the tale, the shall be proved;
and therefore the dead th
_______________0.4_______________
but one word with one of us? couple it with
something and all for his patience.

capulet:
he would not fair come had and stand a pair,
that state to contain whom the mown,
when then i may shall be the fools from the bow.

friar laurence:
why, be the man so but what i have a mooth.

capulet:
what! what thou command is thy tide in my dead.

duke of y
_______________0.6_______________
ill speak more in a minute than he will stand
to i will be the country's grant's too foe;
the state, he do it thou art we would bescase.

coriolanus:
if they not god
it, in thou larted as the all.